# Healthcare KG Population Intelligence — V4.2
**CDC-verified phlebotomy weights • urine albumin method bridging • harmonized race/ethnicity • independent R audit**

**Production safety:** all inputs are read-only, no GitHub API, no writes to the source ZIP, all outputs in `/content/Healthcare_KG_Population_Intelligence_V4_2`. Upload V4.1 output ZIP. This is an **epidemiological screening analysis**, not confirmed CKD, confirmed T2D, drug eligibility, or market size.

**Critical CDC corrections:** For 2021–2023, blood-analyte analyses must use `WTPH2YR` (provided in `BIOPRO_L`), not `WTMEC2YR`. The 2021–2023 urinary albumin method changed; CDC provides a backward Deming bridge to the older scale: `URXUMA_old = -1.643 + 1.189 * URXUMA_new` (mg/L). Because this can be negative at low concentrations, negative predictions are clipped at zero for this *sensitivity* only; primary 2021–2023 results remain unbridged. Urine creatinine requires no correction per CDC. These changes invalidate direct numerical reuse of prior 2021–2023 totals until recomputed.

**CDC sources:**
- https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/overviewbrief.aspx?Cycle=2021-2023
- https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BIOPRO_L.htm
- https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/ALB_CR_L.htm
- https://wwwn.cdc.gov/nchs/nhanes/tutorials/weighting.aspx

**Not a trend test:** Keep survey periods separate; CDC cautions against naively pooling post-pandemic with earlier periods.

In [ ]:
%pip -q install pandas numpy scipy requests pyreadstat matplotlib

## 1. Upload V4.1 outputs ZIP; verify baseline KG integrity

In [ ]:
from pathlib import Path
import os, json, hashlib, zipfile, datetime, io, time
import numpy as np, pandas as pd, requests
from scipy.stats import t as student_t
ROOT=Path('/content/Healthcare_KG_Population_Intelligence_V4_1');ROOT.mkdir(parents=True,exist_ok=True)
UPLOAD_INTERACTIVE=True
INCLUDE_2021_2023=True  # change to False if CDC variable harmonization fails
if UPLOAD_INTERACTIVE:
    from google.colab import files
    uploaded=files.upload()
    incoming=Path(next(iter(uploaded)))
else:
    incoming=Path('/content/v3_1_outputs.zip')
if incoming.suffix.lower()=='.zip':
    with zipfile.ZipFile(incoming) as archive:
        names={Path(x).name:x for x in archive.namelist() if not x.endswith('/')}
        assert 'baseline_kg_COPY.json' in names, 'Missing baseline_kg_COPY.json'
        (ROOT/'baseline_kg_COPY.json').write_bytes(archive.read(names['baseline_kg_COPY.json']))
        prior_audit=next((x for x in names if 'audit' in x.lower() and x.endswith('.json')),None)
        if prior_audit:(ROOT/'v3_1_previous_audit.json').write_bytes(archive.read(names[prior_audit]))
else:
    assert incoming.name=='baseline_kg_COPY.json'
    (ROOT/'baseline_kg_COPY.json').write_bytes(incoming.read_bytes())
kg_path=ROOT/'baseline_kg_COPY.json'
KG_SHA256=hashlib.sha256(kg_path.read_bytes()).hexdigest()
print('Read-only baseline:',kg_path.name,'SHA256:',KG_SHA256)

## 2. Download public NHANES data with SHA-256 provenance

In [ ]:
CYCLES={
 '2015-2016':{'year':'2015','suffix':'I','period_years':2,'era':'pre-pandemic'},
 '2017-2018':{'year':'2017','suffix':'J','period_years':2,'era':'pre-pandemic'},
}
if INCLUDE_2021_2023:
 CYCLES['2021-2023']={'year':'2021','suffix':'L','period_years':2,'era':'post-pandemic-nonstandard'}
COMPONENTS=['DEMO','ALB_CR','BIOPRO','DIQ']
session=requests.Session(); session.headers['User-Agent']='HealthcareKG-V4-research/1.0'
manifest=[]; cycles={}; errors={}
for label,cfg in CYCLES.items():
  tables={}
  try:
    for component in COMPONENTS:
      name=f'{component}_{cfg["suffix"]}'
      url=f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{cfg["year"]}/DataFiles/{name}.xpt'
      response=session.get(url,timeout=120);response.raise_for_status()
      if len(response.content)<1000:raise ValueError('Unexpectedly small XPT: '+name)
      local=ROOT/(name+'.xpt');local.write_bytes(response.content)
      df=pd.read_sas(local,format='xport')
      if 'SEQN' not in df:raise ValueError(f'{name}: SEQN missing; check CDC file/version')
      tables[component]=df
      manifest.append({'cycle':label,'component':name,'url':url,'sha256':hashlib.sha256(response.content).hexdigest(),'n_rows':len(df),'bytes':len(response.content),'status':'OK'})
    cycles[label]=tables
    print(label,[(k,len(v)) for k,v in tables.items()])
  except Exception as exc:
    errors[label]=str(exc)
    print('ERROR:',label,str(exc),'— this cycle will NOT be included')
    if label=='2017-2018':raise
assert '2017-2018' in cycles
pd.DataFrame(manifest).to_csv(ROOT/'v4_source_manifest.csv',index=False)

## 3. Apply CDC-correct weights and harmonize screening variables

In [ ]:
REQUIRED_DEMO=['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU']
REQUIRED_LABS=['URXUMA','URXUCR','LBXSCR']
REQUIRED_DIQ=['DIQ010','DID040']
RACE_MAP={1:'Mexican American',2:'Other Hispanic',3:'Non-Hispanic White',4:'Non-Hispanic Black',6:'Non-Hispanic Asian',7:'Other / multiracial'}
def harmonize(label,tables):
    for name,fields in [('DEMO',REQUIRED_DEMO),('ALB_CR',['URXUMA','URXUCR']),('BIOPRO',['LBXSCR']),('DIQ',REQUIRED_DIQ)]:
        missing=set(fields)-set(tables[name].columns)
        if missing:raise ValueError(f'{label} {name}: missing {sorted(missing)}')
    if label=='2021-2023' and 'WTPH2YR' not in tables['BIOPRO'].columns:
        raise ValueError('CDC-required WTPH2YR absent from BIOPRO_L; STOP, do not substitute MEC weights')
    d=tables['DEMO'][REQUIRED_DEMO].copy()
    for component in ['ALB_CR','BIOPRO','DIQ']:
        d=d.merge(tables[component],on='SEQN',how='left',validate='one_to_one')
    d=d.loc[pd.to_numeric(d.RIDAGEYR,errors='coerce')>=20].copy()
    for col in REQUIRED_DEMO+REQUIRED_LABS+REQUIRED_DIQ:
        d[col]=pd.to_numeric(d[col],errors='coerce')
    d['analysis_weight']=pd.to_numeric(d.WTPH2YR if label=='2021-2023' else d.WTMEC2YR,errors='coerce')
    d['weight_name']='WTPH2YR' if label=='2021-2023' else 'WTMEC2YR'
    female=d.RIAGENDR.eq(2)
    k=np.where(female,.7,.9);a=np.where(female,-.241,-.302)
    ratio=d.LBXSCR/k
    d['egfr']=142*np.minimum(ratio,1)**a*np.maximum(ratio,1)**(-1.2)*(.9938**d.RIDAGEYR)*np.where(female,1.012,1)
    d.loc[d.LBXSCR.isna()|d.LBXSCR.le(0),'egfr']=np.nan
    d['urine_albumin_old_equivalent']=np.maximum(0,-1.643+1.189*d.URXUMA) if label=='2021-2023' else d.URXUMA
    d['uacr']=100*d.URXUMA/d.URXUCR
    d['uacr_bridge']=100*d.urine_albumin_old_equivalent/d.URXUCR
    for name,acol in [('','uacr'),('_bridge','uacr_bridge')]:
        ckd=np.where(d.egfr.notna()&d[acol].notna()&d.URXUCR.gt(0),(d.egfr<60)|(d[acol]>=30),np.nan)
        diabetes=np.where(d.DIQ010.isin([1,2]),d.DIQ010.eq(1).astype(float),np.nan)
        proxy=np.where(pd.Series(diabetes,index=d.index).eq(0),0,np.where(pd.Series(diabetes,index=d.index).eq(1)&d.DID040.between(1,100),(d.DID040>=30).astype(float),np.nan))
        d['joint'+name]=np.where(pd.notna(ckd)&pd.notna(proxy),(ckd==1)&(proxy==1),np.nan)
    d['joint_primary']=d.joint
    d['complete']=d.joint.notna()
    d['case']=d.joint.eq(1).astype(int)
    d['eligible_design']=d.analysis_weight.gt(0)&d.SDMVSTRA.notna()&d.SDMVPSU.notna()
    d['age_group']=pd.cut(d.RIDAGEYR,[20,45,65,75,np.inf],right=False,labels=['20-44','45-64','65-74','75+'])
    d['sex']=d.RIAGENDR.map({1:'Male',2:'Female'})
    d['race_ethnicity_code']=d.RIDRETH3.astype('Int64').astype(str)
    d['race_ethnicity']=d.RIDRETH3.map(RACE_MAP).fillna('Unknown')
    d['cycle']=label
    return d
cohorts={}
for label,tables in cycles.items():
    try:
        cohorts[label]=harmonize(label,tables)
        z=cohorts[label]
        print(label,'weight:',z.weight_name.iloc[0],'adults:',len(z),'evaluable:',int((z.complete&z.eligible_design).sum()),'cases:',int((z.complete&z.eligible_design&z.case.eq(1)).sum()))
    except Exception as e:
        errors[label]=str(e)
        print('STOP:',label,e)
        if label=='2021-2023':raise
assert '2017-2018' in cohorts

## 4. Design-based estimates and method-bridge sensitivity (separate cycles; never pool)

In [ ]:
def survey_total_var(df,quantity):
    x=df[['SDMVSTRA','SDMVPSU']].copy()
    x['v']=df.analysis_weight.to_numpy()*np.asarray(quantity,dtype=float)
    sums=x.groupby(['SDMVSTRA','SDMVPSU'],observed=True).v.sum()
    var=0.
    for _,g in sums.groupby(level=0):
        a=g.to_numpy();m=len(a)
        if m<2:raise ValueError('Lonely PSU; cannot estimate design variance')
        var+=m/(m-1)*np.square(a-a.mean()).sum()
    return float(var)
def survey_stats(d,case_mask,denom_mask):
    z=d.loc[d.eligible_design].copy()
    c=np.asarray(case_mask.reindex(z.index).fillna(False),dtype=float)
    den=np.asarray(denom_mask.reindex(z.index).fillna(False),dtype=float)
    w=z.analysis_weight.to_numpy(float)
    count=float(np.sum(w*c));tot=float(np.sum(w*den))
    dfs=z.groupby('SDMVSTRA',observed=True).SDMVPSU.nunique()
    dfree=int(np.maximum(dfs-1,0).sum())
    if not tot or not dfree:raise ValueError('Invalid denominator or design df')
    p=count/tot;vcount=survey_total_var(z,c);vp=survey_total_var(z,c-p*den)/tot**2
    tcrit=student_t.ppf(.975,dfree)
    return dict(weighted_cases=count,weighted_denominator=tot,prevalence=p,count_variance=vcount,prevalence_variance=vp,
                count_ci_low=max(0,count-tcrit*np.sqrt(vcount)),count_ci_high=count+tcrit*np.sqrt(vcount),
                prevalence_ci_low=max(0,p-tcrit*np.sqrt(vp)),prevalence_ci_high=min(1,p+tcrit*np.sqrt(vp)),
                design_df=dfree,unweighted_cases=int(c.sum()),unweighted_denominator=int(den.sum()))
rows=[]
for label,d in cohorts.items():
    for scenario,indicator in [('primary','joint'),('albumin_bridge','joint_bridge')] if label=='2021-2023' else [('primary','joint')]:
        complete=d[indicator].notna()&d.eligible_design
        case=complete&d[indicator].eq(1)
        rows.append({'cycle':label,'scenario':scenario,'weight':d.weight_name.iloc[0],**survey_stats(d,case,complete)})
main=pd.DataFrame(rows)
main.to_csv(ROOT/'v4_2_corrected_population_estimates.csv',index=False)
display(main[['cycle','scenario','weight','weighted_cases','prevalence','count_ci_low','count_ci_high','unweighted_cases']])

## 5. Independent R `survey` verification of corrected primary estimates

In [ ]:
import shutil, subprocess
if shutil.which('Rscript') is None:
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','r-base-core','r-cran-survey'],check=True)
check=subprocess.run(['Rscript','-e','stopifnot(requireNamespace("survey",quietly=TRUE))'],capture_output=True,text=True)
if check.returncode:raise RuntimeError('R survey unavailable; statistical validation cannot PASS: '+check.stderr[-400:])
r_script=r'''suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu="fail")
a=commandArgs(trailingOnly=TRUE)
d=read.csv(a[1]); D=svydesign(ids=~SDMVPSU,strata=~SDMVSTRA,weights=~analysis_weight,nest=TRUE,data=d)
N=svytotal(~case_indicator,D);P=svyratio(~case_indicator,~denominator_indicator,D)
write.csv(data.frame(weighted_cases=as.numeric(coef(N)[1]),count_variance=as.numeric(vcov(N)[1,1]),prevalence=as.numeric(coef(P)[1]),prevalence_variance=as.numeric(vcov(P)[1,1]),design_df=degf(D)),a[2],row.names=FALSE)
'''
(ROOT/'v4_2_r_validation.R').write_text(r_script)
checks=[]
for label,d in cohorts.items():
    z=d.loc[d.eligible_design,['SDMVSTRA','SDMVPSU','analysis_weight','complete','case']].copy()
    z['case_indicator']=(z.complete&z.case.eq(1)).astype(int)
    z['denominator_indicator']=z.complete.astype(int)
    ip=ROOT/f'v4_2_r_input_{label}.csv';op=ROOT/f'v4_2_r_result_{label}.csv';z.to_csv(ip,index=False)
    run=subprocess.run(['Rscript',str(ROOT/'v4_2_r_validation.R'),str(ip),str(op)],capture_output=True,text=True)
    if run.returncode:checks.append({'cycle':label,'status':'FAIL','error':run.stderr[-500:]});continue
    r=pd.read_csv(op).iloc[0];p=main.loc[main.cycle.eq(label)&main.scenario.eq('primary')].iloc[0]
    rec={'cycle':label,'status':'PASS'}
    for key in ['weighted_cases','count_variance','prevalence','prevalence_variance']:
        rec['relative_error_'+key]=abs(float(p[key])-float(r[key]))/max(abs(float(r[key])),1e-12)
        if rec['relative_error_'+key]>1e-5:rec['status']='FAIL'
    if int(p.design_df)!=int(r.design_df):rec['status']='FAIL'
    checks.append(rec)
r_checks=pd.DataFrame(checks)
r_checks.to_csv(ROOT/'v4_2_r_crosscheck.csv',index=False)
display(r_checks)

## 6. Missing-data response adjustment: recompute under corrected weights (sensitivity only)

In [ ]:
def response_total(z,weights):
    w=np.asarray(weights,float)
    response=np.full(len(z),np.nan)
    fallback=np.full(len(z),-1,dtype=int)
    for level,cols in enumerate([['age_group','sex','race_ethnicity_code'],['age_group','sex'],[]]):
        tmp=z[cols].copy() if cols else pd.DataFrame(index=z.index)
        tmp['w']=w;tmp['observed_w']=w*z.complete.to_numpy().astype(float)
        tmp['observed_n']=(z.complete.to_numpy()&(w>0)).astype(int)
        if cols:
            g=tmp.groupby(cols,observed=True,dropna=False)
            total=g.w.transform('sum').to_numpy();observed=g.observed_w.transform('sum').to_numpy();count=g.observed_n.transform('sum').to_numpy()
        else:
            total=np.full(len(z),tmp.w.sum());observed=np.full(len(z),tmp.observed_w.sum());count=np.full(len(z),tmp.observed_n.sum())
        ok=(count>=10)&(observed>0)&(total>0)
        fill=np.isnan(response)&ok
        response[fill]=observed[fill]/total[fill];fallback[fill]=level
    if np.isnan(response).any():raise ValueError('Unresolved propensity cells')
    return float(np.sum(w*z.complete.to_numpy()*z.case.to_numpy()/response)),float(response.min())
adjusted=[]
for label,d in cohorts.items():
    z=d.loc[d.eligible_design].copy().reset_index(drop=True)
    w=z.analysis_weight.to_numpy(float)
    full,min_response=response_total(z,w)
    reps=[]
    for h,g in z.groupby('SDMVSTRA',observed=True):
        psus=sorted(g.SDMVPSU.dropna().unique());m=len(psus)
        if m<2:raise ValueError(f'{label}: lonely PSU in stratum {h}')
        for j in psus:
            wr=w.copy();same=z.SDMVSTRA.eq(h).to_numpy();drop=same&z.SDMVPSU.eq(j).to_numpy()
            wr[same]*=m/(m-1);wr[drop]=0
            est,_=response_total(z,wr)
            reps.append((h,j,m,est))
    rr=pd.DataFrame(reps,columns=['stratum','psu','m','estimate'])
    v=float(sum((g.m.iloc[0]-1)/g.m.iloc[0]*np.square(g.estimate-g.estimate.mean()).sum() for _,g in rr.groupby('stratum')))
    dfree=int(sum(z.groupby('SDMVSTRA').SDMVPSU.nunique()-1))
    tcrit=student_t.ppf(.975,dfree)
    adjusted.append({'cycle':label,'weight':d.weight_name.iloc[0],'adjusted_count':full,'jackknife_se':np.sqrt(v),'ci_low':max(0,full-tcrit*np.sqrt(v)),'ci_high':full+tcrit*np.sqrt(v),'min_response_probability':min_response,'n_replicates':len(rr)})
adjusted_df=pd.DataFrame(adjusted);adjusted_df.to_csv(ROOT/'v4_2_adjusted_jackknife.csv',index=False)
display(adjusted_df)

## 7. Race/ethnicity harmonization and laboratory comparability decision

In [ ]:
segments=[]
for label,d in cohorts.items():
    for group,z in d.groupby('race_ethnicity',observed=True):
        case=d.index.isin(z.index)&d.complete&d.case.eq(1)
        denom=d.index.isin(z.index)&d.complete
        if not (denom&d.eligible_design).any():continue
        s=survey_stats(d,case,denom)
        segments.append({'cycle':label,'race_ethnicity':group,'reliability':'CAUTION_SMALL_N' if s['unweighted_cases']<30 else 'REVIEW_CI',**s})
seg=pd.DataFrame(segments);seg.to_csv(ROOT/'v4_2_race_ethnicity_segments.csv',index=False)
bridge=main.loc[main.cycle.eq('2021-2023')]
comparison={'phlebotomy_weight':'VERIFIED_WTPH2YR_2021_2023','urine_albumin':'METHOD_CHANGED_BRIDGE_SENSITIVITY_INCLUDED',
 'urine_creatinine':'CDC_NO_CORRECTION_RECOMMENDED','serum_creatinine':'LAB_METHOD_COMPARABILITY_REVIEW_REMAINS',
 'race_ethnicity':'RIDRETH3_LABELS_HARMONIZED_BUT_PRECISION_DIFFERS','trend_claims':'BLOCKED_UNTIL_FULL_REVIEW'}
(ROOT/'v4_2_cdc_methodology_audit.json').write_text(json.dumps(comparison,indent=2))
display(seg[['cycle','race_ethnicity','weighted_cases','unweighted_cases','reliability']])

## 8. Baseline checksum, validation gates and output ZIP

In [ ]:
kg_intact=hashlib.sha256(kg_path.read_bytes()).hexdigest()==kg_hash_initial
r_pass=len(r_checks)==len(cohorts) and r_checks.status.eq('PASS').all()
report={'version':'V4.2','timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'baseline_sha256':kg_hash_initial,'baseline_unchanged':bool(kg_intact),'github_accessed':False,
 'cycles_completed':list(cohorts),'cycle_errors':errors,'independent_r_gate':'PASS' if r_pass else 'FAIL',
 'cdc_weight_gate':'PASS' if '2021-2023' in cohorts and cohorts['2021-2023'].weight_name.iloc[0]=='WTPH2YR' else 'REVIEW',
 'urine_albumin_bridge_sensitivity':'COMPLETED' if '2021-2023' in cohorts else 'NOT_APPLICABLE',
 'epidemiological_gate':'REVIEW_REQUIRED',
 'caveats':['single-visit CKD screen','probable T2D proxy','response adjustment assumes missing-at-random within cells',
 'albumin bridging is sensitivity not clinical correction','serum creatinine cross-period methods not fully audited',
 'no trend or treatment eligibility conclusions']}
(ROOT/'v4_2_validation_report.json').write_text(json.dumps(report,indent=2,default=str))
import zipfile
output_zip=ROOT/'Healthcare_KG_Population_Intelligence_V4_2_Outputs.zip'
with zipfile.ZipFile(output_zip,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT.glob('v4_2_*')):
        if path.is_file() and not path.name.startswith('v4_2_r_input_'):archive.write(path,arcname=path.name)
    archive.write(kg_path,arcname='baseline_kg_COPY.json')
print(json.dumps(report,indent=2))
print('OUTPUT:',output_zip)
assert kg_intact,'Baseline KG modified'

## 9. Download output ZIP

In [ ]:
from google.colab import files
files.download(str(output_zip))

### Decision gate for V5
**Statistical validation** may pass if Python and R agree. **Epidemiological review remains open** until serum creatinine assay comparability, albumin-bridge behavior around UACR=30 mg/g, and missing-data assumptions are reviewed. Do not treat different-period estimates as an established temporal trend.